In [1]:
import pandas as pd
import glob
import numpy as np
import os 
import sys


In [11]:
masterDir = "/Users/danielruiz/Downloads/Alkenes/methodMap1/deltaDir"

fukuiDir = "/Volumes/KINGSTON/fukui"

fukuiDFs = glob.glob(fukuiDir + "/*.csv")

fukuiDFs = [pd.read_csv(f) for f in fukuiDFs]
combinedFukuiDF = pd.concat(fukuiDFs, axis=0, ignore_index=True, join="inner")
print(combinedFukuiDF[:100])

                                               SMILES  \
0        O[C@@H]1C=Cc2c(cc3ccc4cccc5ccc2c3c45)[C@H]1O   
1                          COCc1cc(Br)nc(C2=CCOCC2)c1   
2                   CC1=CCC(COS(=O)(=O)c2ccc(C)cc2)C1   
3   C[C@]12CC[C@H]3C(=C1CCC2=O)CC[C@H]1CC(=O)CC[C@...   
4   COC(=O)C#C[C@@]1(Cc2cccc(OC)c2)C[C@H]2C=C[C@@H...   
..                                                ...   
95                              C=CC1CCCCN1S(C)(=O)=O   
96  O=C(N1CC/C=C\[C@@H](OCc2ccccc2)[C@H](OCc2ccccc...   
97  CC(=O)c1c(OC(=O)c2ccccc2)cc(OC(=O)c2ccccc2)c2c...   
98                        CC(C)=CCCC(C)CCOc1ccc(F)cc1   
99                              C=CCSc1ccc(OCC#CC)cc1   

                                           Canonicals  C1_fuk_neg  C1_fuk_pos  \
0        O[C@@H]1C=Cc2c(cc3ccc4cccc5ccc2c3c45)[C@H]1O    0.047010    0.045356   
1                          COCc1cc(Br)nc(C2=CCOCC2)c1    0.024445    0.002145   
2                   CC1=CCC(COS(=O)(=O)c2ccc(C)cc2)C1    0.053464    0.0

In [12]:
print(list(combinedFukuiDF.columns))

['SMILES', 'Canonicals', 'C1_fuk_neg', 'C1_fuk_pos', 'C1_fuk_neut', 'C2_fuk_neg', 'C2_fuk_pos', 'C2_fuk_neut', 'delta_fuk_neg', 'delta_fuk_pos', 'delta_fuk_neut']


In [13]:
def convertCanonical(str):
    from rdkit import Chem
    mol = Chem.MolFromSmiles(str)
    canonical = Chem.MolToSmiles(mol, isomericSmiles=True, canonical=True)
    return canonical
def fukuiTransfer(inputDF, masterDF, dropCols: list):
    newCols = [col for col in masterDF.columns if col not in dropCols]
    for col in newCols:
        inputDF[col] = "nan"
    for index, row in inputDF.iterrows():
        smiles = row["SMILES"]
        canonical = convertCanonical(smiles)
        if canonical in masterDF["Canonicals"].values:
            idx = masterDF.index[masterDF["Canonicals"] == canonical].tolist()[0]
            matchingRow = masterDF.loc[idx]
            for col in newCols:
                inputDF.at[index, col] = matchingRow[col]
    return inputDF


In [14]:
dirs = glob.glob(masterDir + "/*.csv")
outputDir = "/Volumes/KINGSTON/fukui/mulliken"
for dir_ in dirs:
    df = pd.read_csv(dir_)
    newDF = fukuiTransfer(df , combinedFukuiDF , ["SMILES" , "Canonicals"])
    saveStr = dir_.split("/")[-1].split(".")[0]
    df.to_csv(outputDir + "/" + saveStr + ".csv", index=False)
